Import and load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
df["species_name"] = df["species"].map(dict(enumerate(iris.target_names)))

display(df.head())

EDA (basic checks)

In [ ]:
print("Shape:", df.shape)
print("\nColumn info:")
df.info()

print("\nSummary statistics:")
display(df.describe())

print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nClass counts:\n", df["species_name"].value_counts())

EDA (plots)

In [ ]:
features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

# Distribution of each feature
df[features].hist(figsize=(10, 6), bins=20)
plt.tight_layout()
plt.show()

# Outlier check
df[features].plot(kind="box", figsize=(8, 4))
plt.show()

# Relationships between features, coloured by species
sns.pairplot(df, vars=features, hue="species_name")
plt.show()

# Correlation between features
plt.figure(figsize=(6, 4))
sns.heatmap(df[features].corr(), annot=True, cmap="coolwarm")
plt.show()

Split the data

In [ ]:
from sklearn.model_selection import train_test_split

# --- Classification data: predict species from all 4 features ---
x = df[features]
y = df["species"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# --- Regression data: predict petal_length from the other 3 features ---
x_reg = df[["sepal_length", "sepal_width", "petal_width"]]
y_reg = df["petal_length"]
xr_train, xr_test, yr_train, yr_test = train_test_split(
    x_reg, y_reg, test_size=0.2, random_state=42
)

print("Classification:", x_train.shape, x_test.shape)
print("Regression    :", xr_train.shape, xr_test.shape)

Linear Regression (regression)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lin = LinearRegression()
lin.fit(xr_train, yr_train)

pred = lin.predict(xr_test)

print("Coefficients:", dict(zip(xr_train.columns, lin.coef_.round(3))))
print("Intercept   :", round(lin.intercept_, 3))
print("MAE:", mean_absolute_error(yr_test, pred))
print("R2 :", r2_score(yr_test, pred))

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

log = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200))
log.fit(x_train, y_train)

pred = log.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

K-Nearest Neighbors (KNN)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(x_train, y_train)

pred = knn.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

nb = GaussianNB()
nb.fit(x_train, y_train)          # no scaling needed

pred = nb.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(x_train, y_train)        # no scaling needed

pred = tree.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

print(export_text(tree, feature_names=list(x_train.columns)))

plt.figure(figsize=(12, 6))
plot_tree(tree, feature_names=x_train.columns,
          class_names=iris.target_names, filled=True)
plt.show()

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(x_train, y_train)          # no scaling needed

pred = rf.predict(x_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names))

importance = pd.Series(rf.feature_importances_, index=x_train.columns)
print(importance.sort_values(ascending=False))

Compare all classifiers

In [ ]:
models = {
    "Logistic": log,
    "KNN": knn,
    "NaiveBayes": nb,
    "DecisionTree": tree,
    "RandomForest": rf,
}

results = {name: accuracy_score(y_test, m.predict(x_test)) for name, m in models.items()}
print(pd.Series(results).sort_values(ascending=False))